# Phase 1: Mozilla Common Voice Santali (sat_Olck) Audio Preprocessor
### Edge-Native Vernacular Pedagogy Pipeline for Piper TTS / VITS Voice Synthesis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshrafGalaxy/Vernacular_Pedagogy/blob/main/notebooks/colab_phase1_audio_prep.ipynb)

**Objective:** Download, validate, normalize, and batch-transcode Mozilla Common Voice Santali v26.0 speech clips into standardized **16 kHz 16-bit Mono PCM WAV** with Piper TTS LJSpeech-formatted `metadata.csv` (`clip_id|transcript`), ensuring zero heavy local compute strain.

## 1. Environment Setup & Tool Installation

In [ ]:
# Install FFmpeg and parallel audio processing dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg sox parallel
!pip install -q soundfile tqdm

## 2. Clone Project Repository & Workspace Layout

In [ ]:
import os

# Clone or pull latest main repository
REPO_URL = "https://github.com/AshrafGalaxy/Vernacular_Pedagogy.git"
PROJECT_DIR = "/content/Vernacular_Pedagogy"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin main

%cd {PROJECT_DIR}
os.makedirs("data/raw/common_voice_sat", exist_ok=True)
os.makedirs("data/processed/voice_bank/wavs", exist_ok=True)

## 3. Ingest Common Voice Santali Dataset
*(Upload tar/tsv or fetch directly from Mozilla Data Collective / Hugging Face)*

In [ ]:
import os

# Check if dataset is mounted from Google Drive or uploaded directly
LOCAL_TSV = "data/raw/common_voice_sat/validated.tsv"
CLIPS_DIR = "data/raw/common_voice_sat/clips"

if not os.path.exists(LOCAL_TSV):
    print("Please upload your cv-corpus-*.tar.gz or place validated.tsv and clips/ into data/raw/common_voice_sat/")
    print("You can also mount Google Drive:")
    print("from google.colab import drive; drive.mount('/content/drive')")
else:
    print("Common Voice dataset detected:", LOCAL_TSV)
    print("Clips directory:", CLIPS_DIR)

## 4. Run Santali Ol Chiki Normalizer & Piper Metadata Formatter

In [ ]:
!python scripts/02_audio_common_voice_prep.py \
    --tsv data/raw/common_voice_sat/validated.tsv \
    --output data/processed/voice_bank/metadata.csv

## 5. Parallel Batch Audio Transcoding (16 kHz Mono 16-bit PCM WAV)
Uses multi-core FFmpeg with Linux `xargs` / `parallel` for ultra-fast conversion.

In [ ]:
%%bash
# Find all MP3 files and convert them into 16kHz Mono 16-bit PCM WAV in parallel
INPUT_DIR="data/raw/common_voice_sat/clips"
OUTPUT_DIR="data/processed/voice_bank/wavs"

if [ -d "$INPUT_DIR" ]; then
    echo "Starting parallel transcoding..."
    find "$INPUT_DIR" -name "*.mp3" | parallel -j $(nproc) \
        ffmpeg -y -v error -i {} -acodec pcm_s16le -ac 1 -ar 16000 "$OUTPUT_DIR/{/.}.wav"
    echo "Transcoding complete! Total WAVs: $(ls -1 $OUTPUT_DIR/*.wav 2>/dev/null | wc -l)"
else
    echo "Clips directory $INPUT_DIR not found yet."
fi

## 6. Verification & Quality Assurance Inspection
Inspect audio sample rates, durations, and matching metadata pairs.

In [ ]:
import os
import csv
import soundfile as sf

meta_path = "data/processed/voice_bank/metadata.csv"
wav_dir = "data/processed/voice_bank/wavs"

if os.path.exists(meta_path) and os.path.exists(wav_dir):
    matched = 0
    total_duration_sec = 0.0
    
    with open(meta_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="|")
        for row in reader:
            if not row: continue
            clip_id = row[0]
            wav_file = os.path.join(wav_dir, f"{clip_id}.wav")
            if os.path.exists(wav_file):
                matched += 1
                info = sf.info(wav_file)
                assert info.samplerate == 16000, f"Invalid rate: {info.samplerate}"
                assert info.channels == 1, f"Invalid channels: {info.channels}"
                total_duration_sec += info.duration
                
    print(f"Audio QA Verified:")
    print(f"  - Validated & Transcoded Pairs: {matched}")
    print(f"  - Total Speech Hours:           {total_duration_sec / 3600:.2f} hrs ({total_duration_sec:.1f} s)")
else:
    print("Run steps 3-5 once data is loaded.")

## 7. Package & Export Voice Bank Artifact
Compress the voice bank for training Piper TTS.

In [ ]:
%%bash
# Package standardized dataset into tar.gz
cd data/processed/voice_bank
if [ -f "metadata.csv" ] && [ -d "wavs" ]; then
    tar -czf /content/santali_piper_voicebank_16k.tar.gz metadata.csv wavs/
    ls -lh /content/santali_piper_voicebank_16k.tar.gz
    echo "Voice bank archive ready for Piper TTS training!"
fi